# TMDWF N-State Fit Template

This notebook is a template wrapper around the repository's TMDWF ratio-fit workflow.
Edit the input block below, validate it, and then run the same backend used by the CLI.


## Imports / Setup

Run this notebook from the repository root, or adjust `REPO_ROOT` below.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from lqcd_analysis.notebook_workflows import (
    pretty_print_config,
    render_tmdwf_fit_input_text,
    run_tmdwf_fit_from_notebook,
    validate_tmdwf_notebook_config,
)


## User Inputs

These fields mirror the plain-text TMDWF input file format.
This template is structurally complete, but you should point it at your own HDF5 data and two-point n-state fit tables.


In [ ]:
EXAMPLE_C2PT = REPO_ROOT / "examples" / "data" / "l64c64a076_m140" / "comb_c2pt_csv"
EXAMPLE_qTMDWF = REPO_ROOT / "examples" / "data" / "l64c64a076_m140" / "comb_qTMDWF"
EXAMPLE_2PT_RESULTS = REPO_ROOT / "examples" / "outputs" / "tmdwf_two_point_fit_ref"
EXAMPLE_OUTPUTS = REPO_ROOT / "examples" / "outputs" / "tmdwf_fit_notebook"

workflow_config = {
    # Data settings
    "title_pattern": "l64c64a076_m140_fit_pz*",
    "ns": 64,
    "nt": 64,
    "lattice_spacing_fm": 0.076,
    "decay_constant_check": False,
    "pzlist": [0],
    "gmlist": ["T5"],  # use ["Z5"] for gamma_z gamma_5
    "etalist": ["eta0"],
    "Tdirlist": ["b_X", "b_Y"],
    "bTlist": [0],
    "bzlist": [0],
    "qtmdwf_h5": str(EXAMPLE_qTMDWF / "qTMDWF_CG_1HYP_M140_GSRC_W52_k0_src5_O{gm}.h5"),
    "dataset_path_template": "SP/{gm}/PX0PY0PZ{pz}/{Tdir}/{eta}/bT{bT}/bz{bz}",
    "tsrange": [0, 20],

    # Two-point correlator input
    "two_point_fit_root": str(EXAMPLE_2PT_RESULTS),
    "two_point_fit_window_by_pz": {0: [4, 12]},
    "c2pt": str(EXAMPLE_C2PT / "c2pt_5_5_k0_pz*_real.csv"),
    "fold_t": "periodic",

    # TMDWF fit settings
    "fit_target": "ratio",
    "fit_component": "both",
    "nstates": [1, 2],
    "binsize": 1,
    "bootstrap_samples": 32,
    "bootstrap_size": 32,
    "seed": 2026,
    # Recommended default: one trusted fit window per momentum.
    "fit_window": {0: [4, 12]},

    # This notebook intentionally keeps the window choice explicit.
    "plot": False,  # set True to also write ratio-vs-fit PDFs

    "results_dir": str(EXAMPLE_OUTPUTS),
}


## Option Guide

Edit only `workflow_config` in the cell above for normal usage.
The keys are grouped by comments so data settings, fit settings, and output settings stay easy to scan.

- `title_pattern`: Output title pattern. Use `*` where the momentum index `pz` should be inserted.
- `ns`, `nt`: Spatial and temporal lattice extents.
- `lattice_spacing_fm`: Stored in metadata and summaries.
- `decay_constant_check`: When true, ignore `bTlist` and `bzlist`, fit only the real part at `bT = bz = 0`, and scan windows around the requested `fit_window`. The ground-state matrix element is reported in GeV.
- `fit_target`: Keep this as `"ratio"` in the first implementation.
- `fit_component`: Choose `"real"`, `"imag"`, or `"both"`. The decay-constant check mode always uses `"real"`.
- `nstates`: Supported values are `1`, `2`, or `[1, 2]`.
- `pzlist`: Integer momentum labels to analyze.
- `gmlist`, `etalist`, `Tdirlist`: Lists passed into the HDF5 dataset-path expansion. Supported labels in the first version are `"T5"` for gamma_t gamma_5 and `"Z5"` for gamma_z gamma_5.
- `bTlist` / `bTrange`: Transverse-separation choices. Provide one style only.
- `bzlist` / `bzrange`: Longitudinal-separation choices. When `bz != 0`, the backend combines `+bz` and `-bz` automatically.
- `qtmdwf_h5`: HDF5 file path template for the TMDWF inputs. Use `{gm}` to keep the operator label variable, for example `/path/to/qTMDWF_..._O{gm}.h5`.
- `dataset_path_template`: HDF5 dataset template with placeholders `{gm}`, `{eta}`, `{pz}`, `{Tdir}`, `{bT}`, and `{bz}`.
- `tsrange`: Optional raw time range kept before fitting. If omitted, the backend defaults to `[0, Nt//2 - 1]`.
- `two_point_fit_root`: Root directory that contains the two-point n-state fit outputs. The backend looks under `<two_point_fit_root>/<title>/tables/` and selects the matching `_tmax<tmax>_fits.txt` file.
- `two_point_fit_window_by_pz`: Per-momentum dictionary that tells the backend which two-point `tmin` and `tmax` to reuse for each `pz`. In this notebook, write it as a Python dict such as `{0: [4, 12], 2: [5, 13]}`; the notebook helper materializes it into the tiny three-column mapping file expected by the parser.
- `c2pt`: Two-point correlator CSV used for the denominator of the ratio.
- `fold_t`: Folding mode for the denominator correlator. Use the same convention as the matching two-point analysis.
- Operator behavior: `T5` keeps the original sign-pattern preprocessing and numerator model. `Z5` multiplies the correlator by `-i` before folding and uses the extra lattice-momentum factor `Pz/E_i` in the numerator model.
- `binsize`, `bootstrap_samples`, `bootstrap_size`, `seed`: Bootstrap controls.
- `fit_window`: The canonical window control. Use a dictionary like `{5: [6, 12], 6: [6, 12]}` to set one `[tmin, tmax]` window per momentum. A nested form like `{\"T5\": {5: [6, 12]}}` is also supported for `gm`-specific windows. In decay-constant-check mode, the backend scans `tmin-2` to `tmin+2` and `tmax-4` to `tmax` around this base window. The notebook helper materializes this into the backend fit-window table format automatically.
- Fit initialization: bootstrap sample fits start from the same plain zero initial guess for each sample; there is no separate mean-fit warm start in this template.
- Output grouping: for a fixed `(title, gm, eta, bT)`, the workflow writes one grouped ratio table containing all `bz` entries. For a fixed `(title, gm, eta, bT, component, nstates)`, it writes one grouped summary / fit / samples / curve file containing all `bz` entries.
- Grouped summaries contain one parseable `begin_bz ...` / `end_bz ...` block per `bz`, along with `two_point_fit_table_resolved`, `two_point_fit_tmax_source`, and `two_point_fit_tmax`. The `chi2_dof` and `pvalue` fields are summarized from the successful bootstrap fits.
- Grouped fit tables include `two_point_fit_tmax` plus the usual fit columns.
- `plot`: Optional boolean. When `true`, also write grouped ratio-vs-fit PDF plots for each `(title, gm, eta, bT, component, nstates)` output.
- `results_dir`: Output directory. If set to `None`, notebook runs default to the notebook directory.


## Validate Config


In [ ]:
parsed = validate_tmdwf_notebook_config(workflow_config)
parsed


## Render Plain-Text Input Preview


In [ ]:
print(render_tmdwf_fit_input_text(workflow_config))


## Run Backend Workflow


In [ ]:
outputs = run_tmdwf_fit_from_notebook(workflow_config)
for output in outputs:
    print(output)


## Config Snapshot


In [ ]:
print(pretty_print_config(workflow_config))
